# Figure Generation — ThermoRG-NN

Generates all paper figures from consolidated experimental data.
Run on Kaggle (CPU or GPU) — no TPU required.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.figsize': (6, 4),
    'figure.dpi': 150,
    'font.family': 'serif'
})
import json, os, sys
sys.path.insert(0, '/kaggle/working/ThermoRG-NN')
from thermorg import compute_J_topo
import torch.nn as nn

OUT = '/kaggle/working/figures'
os.makedirs(OUT, exist_ok=True)
print(f"Output: {OUT}")

---
## Figure 1: J_topo Universality Across Families

Data: phase_a_summary.csv

In [ ]:
# Load Phase A data
import csv
phase_a = []
with open('/kaggle/working/ThermoRG-NN/phase_a_summary.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        phase_a.append(row)

# Group by family
families = {}
for r in phase_a:
    g = r['group']
    families.setdefault(g, []).append({
        'name': r['name'],
        'J_topo': float(r['J_topo']),
        'alpha': float(r['alpha']),
        'beta': float(r.get('beta', 0)),
        'params_M': float(r['params_M'])
    })

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Panel A: J_topo vs alpha
colors = {'G1': 'C0', 'G2': 'C1', 'G3': 'C2', 'G4': 'C3'}
markers = {'G1': 'o', 'G2': 's', 'G3': '^', 'G4': 'D'}
for g, archs in families.items():
    xs = [a['alpha'] for a in archs]
    ys = [a['J_topo'] for a in archs]
    names = [a['name'] for a in archs]
    axes[0].scatter(xs, ys, c=colors[g], marker=markers[g], s=60, label=g, zorder=5)

axes[0].set_xlabel(r'$\alpha$ (pre-asymptotic)')
axes[0].set_ylabel(r'$J_{\mathrm{topo}}$')
axes[0].set_title('(a) J_topo vs alpha')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel B: J_topo distribution by family
for g, archs in families.items():
    ys = [a['J_topo'] for a in archs]
    axes[1].scatter([g]*len(ys), ys, c=colors[g], marker=markers[g], s=60, label=g, zorder=5)

axes[1].set_xlabel('Architecture Family')
axes[1].set_ylabel(r'$J_{\mathrm{topo}}$')
axes[1].set_title('(b) J_topo by family')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/fig1_J_topo_universality.png', bbox_inches='tight')
plt.show()
print('Fig1 saved')

---
## Figure 4: J_topo(D) Log-Log Scaling

Data: CPU validation (5-seed averaged)

In [ ]:
# J_topo(D) data from CPU validation
data_L3 = {
    'D': [16, 24, 32, 48, 64, 96],
    'J': [0.8429, 0.8126, 0.7807, 0.7351, 0.6944, 0.6472],
    'std': [0.016, 0.016, 0.013, 0.015, 0.007, 0.009]
}
data_L5 = {
    'D': [16, 24, 32, 48, 64, 96],
    'J': [0.8684, 0.8653, 0.8442, 0.8149, 0.7768, 0.7515],
    'std': [0.01]*6  # approximate
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, L_data, L in zip(axes, [data_L3, data_L5], [3, 5]):
    D = np.array(L_data['D'])
    J = np.array(L_data['J'])
    std = np.array(L_data['std'])
    
    # Fit
    log_D = np.log(D)
    log_J = np.log(J)
    slope, intercept = np.polyfit(log_D, log_J, 1)
    
    ax.errorbar(D, J, yerr=std, fmt='o', capsize=3, s=60, zorder=5)
    
    # Theory line
    alpha_GELU = 0.45
    D_fit = np.linspace(10, 120, 200)
    J_theory = np.exp(intercept) * D_fit**(slope)
    ax.plot(D_fit, J_theory, '--', color='gray', label=f'fit: slope={slope:.3f}')
    
    # Predicted
    J_pred = J[2] * (D_fit / D[2])**(-alpha_GELU/L)
    ax.plot(D_fit, J_pred, ':', color='red', alpha=0.7, label=f'theory: slope={-alpha_GELU/L:.3f}')
    
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'$D$ (width)'); ax.set_ylabel(r'$J_{\mathrm{topo}}$')
    ax.set_title(f'L={L}: slope={slope:.3f} (theory={-alpha_GELU/L:.3f})')
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_xlim(12, 120)

plt.suptitle(r'$J_{\mathrm{topo}}(D) \propto D^{-\alpha_{\mathrm{GELU}}/L}$', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_J_topo_D_scaling.png', bbox_inches='tight')
plt.show()
print('Fig4 saved')

---
## Figure 5: β(γ) Cooling Theory

Data: Phase S1 (BN, None) + Phase B1 (LN)

In [ ]:
gamma_vals = np.array([3.39, 2.29, 0.41])
beta_vals  = np.array([1.117, 0.950, 0.219])
norms = ['None', 'BN', 'LN']
colors_ = ['red', 'orange', 'blue']

g_curve = np.linspace(0.3, 4.0, 300)
b_curve = 0.425 * np.log(g_curve / 2.0) + 0.893

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(g_curve, b_curve, 'k--', alpha=0.5,
        label=r'$\beta(\gamma) = 0.425\ln(\gamma/2.0) + 0.893$')

for g, b, n, c in zip(gamma_vals, beta_vals, norms, colors_):
    ax.scatter(g, b, s=100, color=c, zorder=5, label=n)
    ax.annotate(n, (g, b), xytext=(8, 0), textcoords='offset points', fontsize=11)

ax.axvline(2.0, color='gray', linestyle=':', alpha=0.7, label=r'$\gamma_c = 2.0$')
ax.set_xlabel(r'$\gamma$ (variance fluctuation)')
ax.set_ylabel(r'$\beta$ (scaling exponent)')
ax.set_title('Cooling Theory: β(γ) Validated for γ ∈ [0.41, 3.39]')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 4); ax.set_ylim(0, 1.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_beta_gamma_cooling.png', bbox_inches='tight')
plt.show()
print('Fig5 saved')

---
## Figure 9: LN Scaling Law (Phase B1)

Data: Phase B1, 8 runs × 200 epochs

In [ ]:
from scipy.optimize import curve_fit

# LN data (Phase B1)
D_LN = np.array([32, 32, 48, 48, 64, 64, 96, 96])
loss_LN = np.array([1.0958, 1.1056, 1.0212, 1.0274, 0.9626, 0.9757, 0.9035, 0.9045])
D_unique = np.array([32, 48, 64, 96])
loss_avg = np.array([loss_LN[D_LN==d].mean() for d in D_unique])

def plaw(D, a, b, c): return a * D**(-b) + c
popt, _ = curve_fit(plaw, D_unique, loss_avg, p0=[10, 0.5, 0.5],
                     bounds=([0,0,0],[1000,3,3]), maxfev=10000)
a, beta_LN, c = popt
loss_pred = plaw(D_unique, *popt)
r2 = 1 - (((loss_avg - loss_pred)**2).sum()) / (((loss_avg - loss_avg.mean())**2).sum())

D_fit = np.linspace(20, 120, 200)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(D_unique, loss_avg, s=80, zorder=5, label='LN data (avg over 2 seeds)')
ax.plot(D_fit, plaw(D_fit, *popt), 'b-', label=f'fit: β={beta_LN:.3f}, R²={r2:.4f}')
ax.xscale('log'); ax.yscale('log')
ax.set_xlabel(r'$D$ (width)'); ax.set_ylabel('Val Loss')
ax.set_title('LN Cooling Validation (Phase B1)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig9_LN_scaling_law.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig9 saved: β_LN={beta_LN:.3f}, R²={r2:.4f}')

---
## Figure 10: Round 1 — HBO vs Random (Negative Result)

Data: Phase B2, 50 epochs

In [ ]:
# Phase B2 L2 data
random_l2 = [
    (0.3858, 0.7739, 96, 6, False),
    (0.5014, 0.8062, 64, 5, False),
    (0.6047, 0.7838, 64, 5, True),
    (0.6268, 0.7538, 64, 4, False),
    (0.6937, 0.8027, 48, 4, False),
]
hbo_l2 = [
    (0.6051, 0.8627, 32, 6, False),
    (0.6812, 0.8774, 24, 6, False),
    (0.7479, 0.8455, 32, 5, True),
    (0.7821, 0.8727, 24, 6, True),
    (0.8378, 0.8701, 24, 5, True),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Panel A: bar chart of final losses
r_losses = [r[0] for r in random_l2]
h_losses = [r[0] for r in hbo_l2]
x = np.arange(5)
w = 0.35
axes[0].bar(x - w/2, r_losses, w, label='Random', color='steelblue')
axes[0].bar(x + w/2, h_losses, w, label='HBO', color='coral')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Val Loss (50 epochs)')
axes[0].set_title('Random vs HBO: Final Loss')
axes[0].legend()
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'#{i+1}' for i in range(5)])
axes[0].grid(True, alpha=0.3, axis='y')

# Panel B: width distribution
r_widths = [r[2] for r in random_l2]
h_widths = [r[2] for r in hbo_l2]
axes[1].scatter([0]*len(r_widths), r_widths, s=80, c='steelblue', label='Random', zorder=5)
axes[1].scatter([1]*len(h_widths), h_widths, s=80, c='coral', label='HBO', zorder=5)
axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['Random', 'HBO'])
axes[1].set_ylabel('Width')
axes[1].set_title('Architecture: HBO selected narrow-deep')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT}/fig10_HBO_vs_Random.png', bbox_inches='tight')
plt.show()
print('Fig10 saved')

---
## Figure 12: Confounding Analysis (Simpson's Paradox)

Data: Phase B2, n=10

In [ ]:
# Phase B2 data
phase_b2 = [
    (96, 6, False, 0.7739, 0.3858),
    (64, 5, False, 0.8062, 0.5014),
    (64, 5, True,  0.7838, 0.6047),
    (64, 4, False, 0.7538, 0.6268),
    (48, 4, False, 0.8027, 0.6937),
    (32, 6, False, 0.8627, 0.6051),
    (24, 6, False, 0.8774, 0.6812),
    (32, 5, True,  0.8455, 0.7479),
    (24, 6, True,  0.8727, 0.7821),
    (24, 5, True,  0.8701, 0.8378),
]

widths  = np.array([r[0] for r in phase_b2])
j_topo  = np.array([r[3] for r in phase_b2])
losses  = np.array([r[4] for r in phase_b2])

from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel 1: J_topo vs Loss
axes[0].scatter(j_topo, losses, c=widths, cmap='viridis', s=80, zorder=5)
r, p = spearmanr(j_topo, losses)
axes[0].set_xlabel(r'$J_{\mathrm{topo}}$')
axes[0].set_ylabel('Val Loss')
axes[0].set_title(f'J_topo vs Loss\nSimple r={r:+.3f} (p={p:.3f})')
from matplotlib.cm import ScalarMappable
sm = ScalarMappable(cmap='viridis', norm=plt.Normalize(widths.min(), widths.max()))
sm.set_array([])
plt.colorbar(sm, ax=axes[0], label='Width')
axes[0].grid(True, alpha=0.3)

# Panel 2: Width vs Loss
axes[1].scatter(widths, losses, c=j_topo, cmap='plasma', s=80, zorder=5)
r2, p2 = spearmanr(widths, losses)
axes[1].set_xlabel('Width')
axes[1].set_ylabel('Val Loss')
axes[1].set_title(f'Width vs Loss\nr={r2:+.3f} (p={p2:.3f})')
sm2 = ScalarMappable(cmap='plasma', norm=plt.Normalize(j_topo.min(), j_topo.max()))
sm2.set_array([])
plt.colorbar(sm2, ax=axes[1], label=r'$J_{\mathrm{topo}}$')
axes[1].grid(True, alpha=0.3)

# Panel 3: Width vs J_topo (confounder)
axes[2].scatter(widths, j_topo, c=losses, cmap='RdYlGn_r', s=80, zorder=5)
r3, p3 = spearmanr(widths, j_topo)
axes[2].set_xlabel('Width')
axes[2].set_ylabel(r'$J_{\mathrm{topo}}$')
axes[2].set_title(f'Width vs J_topo (Confounder)\nr={r3:+.3f}')
sm3 = ScalarMappable(cmap='RdYlGn_r', norm=plt.Normalize(losses.min(), losses.max()))
sm3.set_array([])
plt.colorbar(sm3, ax=axes[2], label='Loss')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Simpson\'s Paradox: Simple vs Partial Correlation', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT}/fig12_confounding.png', bbox_inches='tight')
plt.show()
print('Fig12 saved')

---
## Figure 13: Partial Correlation Bar Chart

Summary of correlation analysis

In [ ]:
from sklearn.linear_model import LinearRegression
def partial_corr(x, y, z):
    lr_xz = LinearRegression().fit(z.reshape(-1,1), x)
    x_resid = x - lr_xz.predict(z.reshape(-1,1))
    lr_yz = LinearRegression().fit(z.reshape(-1,1), y)
    y_resid = y - lr_yz.predict(z.reshape(-1,1))
    return spearmanr(x_resid, y_resid)

widths = np.array([r[0] for r in phase_b2])
j_topo = np.array([r[3] for r in phase_b2])
losses = np.array([r[4] for r in phase_b2])

r_simple, p_simple = spearmanr(j_topo, losses)
r_part, p_part = partial_corr(j_topo, losses, widths)

fig, ax = plt.subplots(figsize=(5, 4))
labels = ['Simple\nr(J,L)', 'Partial\nr(J,L|W)']
values = [r_simple, r_part]
colors = ['C7', 'C0']
bars = ax.bar(labels, values, color=colors, width=0.5)
ax.axhline(0, color='black', linewidth=0.5)
for bar, v, p in zip(bars, values, [p_simple, p_part]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.03 if v > 0 else v - 0.08,
            f'r={v:+.3f}\np={p:.3f}', ha='center', fontsize=11)
ax.set_ylabel(r'Spearman $r$')
ax.set_title('J_topo → Loss Correlation\n(Controlling for Width)')
ax.set_ylim(-1.1, 1.1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{OUT}/fig13_partial_corr.png', bbox_inches='tight')
plt.show()
print('Fig13 saved')